<style>
.note {padding: 12px 16px; border-left: 5px solid #2563eb; background: #eff6ff; margin: 10px 0;}
.warn {padding: 12px 16px; border-left: 5px solid #d97706; background: #fffbeb; margin: 10px 0;}
.fix  {padding: 12px 16px; border-left: 5px solid #dc2626; background: #fef2f2; margin: 10px 0;}
.exam {padding: 12px 16px; border-left: 5px solid #059669; background: #ecfdf5; margin: 10px 0;}
table {font-size: 95%;}
</style>

# 04 — Z vs t, Errors, Confidence Intervals and Margin of Error

### The decision framework behind mean inference

**Level:** beginner → advanced  
**Style:** short explanations, worked examples, formulas, runnable code, revision material

## What you will be able to do

- select z or t using a statistically correct rule
- distinguish Type I error, Type II error, and power
- calculate and interpret confidence intervals and margin of error
- explain the link between intervals and two-sided tests

## Resource coverage

- Transcript lines 2245–2547: z-versus-t flowchart and Type I/II outcomes
- Transcript lines 2891–3176: point estimates, margin of error, and CAT-score confidence interval
- Handwritten z-vs-t resource: page 1 shows the lecture decision tree
- Handwritten error resource: page 1 shows the four reality/decision outcomes

<div class="note"><b>How to study this notebook:</b> Read once without memorising. Then rerun the code, solve each checkpoint without looking, and finish with the cheat sheet.</div>


## 1. Z or t? Use this rule

| Situation for a mean | Distribution |
|---|---|
| Population SD $\sigma$ genuinely known; normal population or valid large-sample approximation | z |
| Population SD unknown and estimated by sample SD $s$ | t |

The lecture’s flowchart adds “if $n<30$, use t even when $\sigma$ is known.” Treat 30 as a classroom heuristic, not a law.

### More precise view

- If $\sigma$ is known and observations come from a normal population, the z statistic is exact even for a small $n$.
- If $\sigma$ is unknown, the t statistic is correct under normality for any $n$.
- When $n$ is large, t and z give almost identical results.
- Sample size affects the quality of the normal approximation; it is not the fundamental identity of the test.

<div class="note"><b>Memory rule:</b> known sigma → z; estimated sigma → t. Then separately check shape, outliers, independence, and design.</div>


In [1]:
import math
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=4, suppress=True)


In [2]:
# How quickly t critical values approach z = 1.95996 for a 95% interval
rows = []
for df in [2, 5, 10, 29, 100, 1000]:
    rows.append({'df': df, 't_critical_95%': stats.t.ppf(0.975, df)})
print(pd.DataFrame(rows).to_string(index=False))
print(f"z critical: {stats.norm.ppf(0.975):.6f}")


  df  t_critical_95%
   2        4.302653
   5        2.570582
  10        2.228139
  29        2.045230
 100        1.983972
1000        1.962339
z critical: 1.959964


## 2. Decisions versus reality

A statistical procedure can be correct or wrong. Reality is fixed; our sample-based decision is uncertain.

| Reality | Fail to reject $H_0$ | Reject $H_0$ |
|---|---|---|
| $H_0$ true | correct decision | **Type I error** |
| $H_0$ false | **Type II error** | correct detection |

### Type I error

- Reject a true $H_0$.
- False positive.
- Probability: $\alpha=P(\text{reject }H_0\mid H_0\text{ true})$.

### Type II error

- Fail to reject a false $H_0$.
- False negative.
- Probability: $\beta=P(\text{fail to reject }H_0\mid H_0\text{ false at a specified effect})$.

### Power

$$\text{Power}=1-\beta$$

Power is the probability that the procedure detects a specified real effect.


## 3. What controls power?

Power generally rises when:

- sample size increases;
- the true effect is larger;
- measurement noise decreases;
- $\alpha$ is increased;
- a justified one-sided test replaces a two-sided test.

Trade-off:

- Lowering $\alpha$ reduces false positives.
- With everything else fixed, it increases false negatives.
- Increasing sample size can reduce both error risks for meaningful effects.

Power must be calculated for a **specific alternative effect size**. There is no single $\beta$ for “$H_0$ is false.”


In [3]:
# Monte Carlo power: two-sided one-sample z-test
rng = np.random.default_rng(7)
alpha = 0.05
sigma = 10
true_shift = 3
simulations = 20_000

for n in [10, 30, 100]:
    sample_means = rng.normal(loc=true_shift, scale=sigma/math.sqrt(n), size=simulations)
    z = sample_means / (sigma/math.sqrt(n))
    reject = np.abs(z) > stats.norm.ppf(1-alpha/2)
    print(f"n={n:3d}: estimated power = {reject.mean():.3f}")


n= 10: estimated power = 0.155
n= 30: estimated power = 0.373
n=100: estimated power = 0.849


## 4. Point estimate, standard error, margin of error

### Point estimate

A single best estimate, such as $\bar x$ for $\mu$.

### Standard error

The estimated sampling variability of the statistic.

$$SE_{\bar x}=\frac{\sigma}{\sqrt n}\quad\text{or}\quad\frac{s}{\sqrt n}$$

### Margin of error

$$ME=(\text{critical value})\times SE$$

### Confidence interval

$$\text{estimate}\pm ME$$

Known $\sigma$:

$$\bar x\pm z_{1-\alpha/2}\frac{\sigma}{\sqrt n}$$

Unknown $\sigma$:

$$\bar x\pm t_{1-\alpha/2,n-1}\frac{s}{\sqrt n}$$


## 5. Correct interpretation of a 95% confidence interval

Frequentist wording:

> If we repeatedly sampled and built intervals using this procedure, about 95% of those intervals would contain the true parameter.

For the one interval already calculated, the population parameter is fixed. We say we are 95% confident in the **procedure**, not that the fixed parameter has a 95% chance of being in this realised interval.

The interval describes plausible parameter values under the model. It does not say that 95% of observations lie inside it.


## 6. Lecture CAT-score example—and its internal mismatch

Transcript statement:

- known population SD $\sigma=100$;
- sample mean $\bar x=520$;
- sample size stated as $n=30$;
- construct a 95% confidence interval.

Correct calculation using $n=30$:

$$ME=1.96\frac{100}{\sqrt{30}}=35.78$$

$$CI=(520-35.78,520+35.78)=(484.22,555.78)$$

The lecture then substitutes $\sqrt{25}$, producing:

$$ME=1.96\frac{100}{5}=39.2,\qquad CI=(480.8,559.2)$$

<div class="fix"><b>Resource correction:</b> 480.8–559.2 is correct only if n = 25. If n = 30 as stated, use 484.22–555.78.</div>


In [4]:
def z_interval(xbar, sigma, n, confidence=0.95):
    alpha = 1 - confidence
    crit = stats.norm.ppf(1 - alpha/2)
    me = crit * sigma / math.sqrt(n)
    return me, (xbar-me, xbar+me)

for n in [30, 25]:
    me, ci = z_interval(520, 100, n)
    print(f"n={n}: ME={me:.2f}, CI=({ci[0]:.2f}, {ci[1]:.2f})")


n=30: ME=35.78, CI=(484.22, 555.78)
n=25: ME=39.20, CI=(480.80, 559.20)


## 7. What changes interval width?

| Change | Effect on width |
|---|---|
| larger confidence level | wider |
| larger sample size | narrower |
| larger SD/noise | wider |
| paired design that removes person-to-person noise | often narrower |

Because $SE\propto1/\sqrt n$, cutting margin of error in half requires about four times the sample size.

Approximate z-based planning formula:

$$n=\left(\frac{z_{1-\alpha/2}\sigma}{ME}\right)^2$$

Round up to the next whole number.


## 8. CI and hypothesis-test equivalence

For a two-sided level-$\alpha$ test using the same assumptions:

- null value outside the $(1-\alpha)$ interval → reject $H_0$;
- null value inside the interval → fail to reject $H_0$.

This is why intervals are more informative:

- the p-value says how incompatible the data are with one null value;
- the interval shows a range of parameter values compatible with the data.

For one-sided tests, use the matching one-sided confidence bound; do not mechanically compare a two-sided 95% interval with every one-sided 5% test.


# End-of-topic cheat sheet

| Concept | Memory line |
|---|---|
| z vs t | known $\sigma$ → z; estimated $s$ → t |
| Type I | reject true $H_0$; probability $\alpha$ |
| Type II | miss a specified false $H_0$; probability $\beta$ |
| Power | $1-\beta$ |
| Standard error | variability of an estimator across samples |
| Margin of error | critical value × SE |
| Confidence interval | estimate ± margin of error |
| Larger $n$ | smaller SE at rate $1/\sqrt n$ |
| 95% two-sided CI | corresponds to a 5% two-sided test |

**Do not confuse:** SD describes spread of observations; SE describes uncertainty in an estimate.


# Revision questions and answers

**Q1. If sigma is unknown and n = 500, which mean test is principled?**

<details><summary>Answer</summary>

A t-test. It will be numerically almost identical to z because df is large.

</details>

---

**Q2. What is a Type I error?**

<details><summary>Answer</summary>

Rejecting $H_0$ when $H_0$ is true.

</details>

---

**Q3. What is a Type II error?**

<details><summary>Answer</summary>

Failing to reject $H_0$ when a specified alternative is true.

</details>

---

**Q4. What is power?**

<details><summary>Answer</summary>

$1-\beta$, the probability of rejecting $H_0$ for a specified real effect.

</details>

---

**Q5. How does quadrupling n affect the standard error?**

<details><summary>Answer</summary>

It halves the standard error because SE scales as $1/\sqrt n$.

</details>

---

**Q6. Does a 95% CI contain 95% of the raw data?**

<details><summary>Answer</summary>

No. It estimates uncertainty in a parameter, not the spread of individual observations.

</details>

---

**Q7. What is the corrected n=30 CAT interval?**

<details><summary>Answer</summary>

Approximately (484.22, 555.78).

</details>

---

**Q8. What does it mean if a 95% CI excludes the null value?**

<details><summary>Answer</summary>

The matching two-sided test rejects at $\alpha=0.05$.

</details>
